# Step 06.5 — Sampling Batches：一次训练到底喂给模型什么？

到现在为止，我们已经会从一个 start position 取出一个长度为 `T+1` 的 chunk，再构造：

```text
X.shape = (T,)
Y.shape = (T,)
```

真正训练时，一次通常不会只用一个窗口，而是同时取多个窗口组成一个 **batch**。

这一节完整解决 5 个问题：

1. `B`（batch size）和 `T`（sequence length）分别控制什么；
2. `get_batch()` 怎样随机选择多个 start position；
3. 一次 batch 在长语料上到底覆盖了哪些位置；
4. seed / RNG state 与 reproducibility 到底是什么关系；
5. 教学 sampler 与真实 LLM 数据管线相比还缺什么。

## 1. Batch Size `B` 与 Sequence Length `T` 是两个完全不同的轴

假设：

```text
batch_size = B = 3
block_size = T = 4
```

那么一次训练会拿 3 个独立的文本窗口，每个窗口有 4 个 input token：

```text
X =
[ window 0: 4 tokens ]
[ window 1: 4 tokens ]
[ window 2: 4 tokens ]

X.shape = (3,4) = (B,T)
```

`B` 回答：**一次并行处理多少条 sequence/window？**

`T` 回答：**每条 sequence 有多少个 token position？**

增大 `B` 不会让单个样本看到更长历史；增大 `T` 才会扩大单个样本的时间长度。

每个长度 `T` 的 causal block 有 `T` 个监督位置，所以一个 batch 一共有 `B*T` 个 token-level target。

例如 `B=3, T=4`，一次 forward 就提供 12 个 next-token supervision positions。

## 2. 写出第一个 `get_batch()`：随机 start → chunk → X/Y → stack

先准备一段足够长、方便肉眼检查的字符语料。这里使用 Python 标准库 `random.Random` 控制 sampler 的随机性；真实大型训练的数据 loader 会更复杂，但这个版本最适合看清核心逻辑。

In [ ]:
import random
import mlx.core as mx

corpus = (
    "abcdefghijklmnopqrstuvwxyz"
    "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    "0123456789"
)

chars = sorted(set(corpus))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
data = mx.array([stoi[ch] for ch in corpus])

def decode(ids):
    return "".join(itos[int(i)] for i in ids)

print("N =", data.shape[0])
print("vocab_size =", len(chars))

Sampler 的核心逻辑是：重复 `B` 次随机选择一个 `start`，取 `T+1` 个连续 token，前 `T` 个作为 X、后 `T` 个作为 Y，最后把多个 `(T,)` 向量 stack 成 `(B,T)`。

合法 start 范围是：

```text
0 <= start < N - T
```

因为 Python `randrange(stop)` 不包含 stop，所以最大 start 正好是 `N-T-1`，还能完整拿到 `T+1` 个 token。

In [ ]:
def get_batch(data, batch_size, block_size, rng):
    N = data.shape[0]

    if N <= block_size:
        raise ValueError("data must contain at least block_size + 1 tokens")

    starts = [
        rng.randrange(0, N - block_size)
        for _ in range(batch_size)
    ]

    xs = []
    ys = []
    for start in starts:
        chunk = data[start : start + block_size + 1]
        xs.append(chunk[:-1])
        ys.append(chunk[1:])

    X = mx.stack(xs)
    Y = mx.stack(ys)
    return X, Y, starts

rng = random.Random(42)
X, Y, starts = get_batch(data, batch_size=3, block_size=4, rng=rng)

print("starts :", starts)
print("X shape:", X.shape)
print("Y shape:", Y.shape)

for b in range(X.shape[0]):
    print(
        f"batch {b}: start={starts[b]:2d}  "
        f"X={decode(X[b].tolist())!r}  "
        f"Y={decode(Y[b].tolist())!r}"
    )

`mx.stack(xs)` 会创建一个新的 Batch axis。每个单独 window 原来是 `(T,)`，把 `B` 个这样的向量堆起来就得到 `(B,T)`。

这和 `reshape` 不一样：`reshape` 只是重新解释同一批元素的 shape；`stack` 是把多个独立 array 组合到一个新的 axis 上。

## 3. 一次随机 Batch 在语料上抽到了哪里？——把 starts 可视化

单看 `starts=[40,7,1,...]` 不够直观。我们把 corpus position 放横轴、batch item 放纵轴，把每个 window 覆盖的位置画出来。

这里继续使用 `imshow()`：矩阵每一行对应一个 batch sample。

In [ ]:
import matplotlib.pyplot as plt

batch_size = 8
block_size = 6
viz_rng = random.Random(7)

X_viz, Y_viz, starts_viz = get_batch(
    data,
    batch_size=batch_size,
    block_size=block_size,
    rng=viz_rng,
)

N = data.shape[0]
coverage = [[0] * N for _ in range(batch_size)]
for b, start in enumerate(starts_viz):
    for pos in range(start, start + block_size):
        coverage[b][pos] = 1

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.imshow(coverage, aspect="auto")
ax.set_yticks(range(batch_size))
ax.set_yticklabels([f"b={b}, start={s}" for b, s in enumerate(starts_viz)])
ax.set_xlabel("corpus position")
ax.set_ylabel("batch item")
ax.set_title("Random windows sampled in one batch")
plt.show()

这张图能看出：同一个 batch 的 sample 可以来自完全不同位置，也可能部分重叠；如果采用有放回采样，甚至可能出现相同 start。

最重要的是区分：

```text
T axis -> 一个 sequence 内部的 token 顺序
B axis -> 多条独立训练 sequence 的集合
```

Batch 第 0 行和第 1 行并不表示文本上前后相接。

## 4. Seed 与 Reproducibility：同一个 seed 到底保证什么？

伪随机数生成器可以理解为：

```text
seed + 当前 RNG state
        ↓
下一批“看起来随机”的数
        ↓
RNG state 继续向前推进
```

两个新创建、seed 相同的 RNG，在调用顺序相同的情况下，会生成相同的 start sequence。

In [ ]:
rng_a = random.Random(123)
rng_b = random.Random(123)

_, _, starts_a = get_batch(data, 4, 5, rng_a)
_, _, starts_b = get_batch(data, 4, 5, rng_b)

print("starts_a:", starts_a)
print("starts_b:", starts_b)
print("same     :", starts_a == starts_b)

但同一个 RNG 连续调用两次，第二次会从已经推进后的 state 继续，所以 batch 通常不同。

In [ ]:
rng = random.Random(123)
_, _, first = get_batch(data, 4, 5, rng)
_, _, second = get_batch(data, 4, 5, rng)

print("first batch starts :", first)
print("second batch starts:", second)
print("same:", first == second)

所以更准确地说：**seed 决定伪随机序列从哪里开始，RNG state 决定当前已经走到哪里。**

这对 checkpoint/resume 很重要。训练到中途若只保存初始 seed、却不恢复当时 RNG/sampler state，后续 batch 顺序通常不会和原训练完全一致。

另外，sampler 可复现并不自动保证所有 GPU/Metal 运算在任何环境下逐 bit 一致；那属于更广义的 determinism 问题，后面工程阶段再讲。

## 5. 为什么随机抽窗口？真实 LLM 数据管线又会复杂到什么程度？

从头到尾顺序读取并不是数学上错误，但教学版随机窗口有几个优点：每个 step 容易接触不同语料区域、减少连续 step 的局部相关性、实现简单，也不需要复杂游标。

因此我们的 toy sampler 使用 **random windows with replacement**。

但真实大模型训练不会只有这一个 `get_batch()`。通常还要处理：

```text
multiple documents / datasets
-> train / validation split
-> tokenization
-> document shuffling
-> packing / padding / boundary handling
-> dataset shards
-> distributed workers / GPUs
-> deterministic sampler state
-> (B,T) batches
```

两个原则现在就要记住：

第一，train / validation 必须逻辑隔离。Validation token 不应该被 training sampler 用来更新 parameter，否则验证结果被污染。

第二，随机本身不是目的。真正目标是得到合理、吞吐高、可恢复、偏差可控的数据流。大型系统可能使用 deterministic shuffle、shard cursor、packing 等，而不是 Python `randrange()`。

### Step 6 的完整数据链现在闭合

```text
raw text
   ↓ tokenizer
token IDs
   ↓ long corpus tensor
N tokens
   ↓ sample B starts
B chunks of length T+1
   ↓ next-token shift
X(B,T), Y(B,T)
   ↓ embedding lookup
X -> (B,T,C)
   ↓ model
logits -> (B,T,V)
   +
Y(B,T)
   ↓
language-model loss
```

### 6.5 最终结论

- `B` 是一次并行的 sequence/window 数，`T` 是每条 sequence 的 token 长度；
- 一个 batch 含 `B*T` 个 token-level next-token targets；
- `get_batch()` 的核心是随机 start、连续取 `T+1`、shift、再 stack；
- `mx.stack` 创建新的 Batch axis，把多个 `(T,)` 变成 `(B,T)`；
- Batch 中各行是独立样本，不代表文本时间上的前后连接；
- 同 seed + 同调用顺序可复现 sampler；严格 resume 还需要恢复 RNG/sampler state；
- toy random-window sampler 只是教学版本，真实 LLM 管线还涉及 split、shuffle、packing、sharding、distributed loading 和 deterministic resume；
- 完成本节后 Step 6 完成。下一步进入 Step 7.1：真正使用 `nn.Embedding`，把 6.2 手工理解的 embedding lookup 接入 MLX Module/Parameter 系统。